<div align='center' style='background:#0a0f1e;padding:30px;border-radius:12px;border:1px solid #1e3a5f'>

# 🛡️ SleepGuard AI
## Real-Time Sleep Detection using Google ADK Agents + Gemini Vision

---

### 🧠 Architecture
```
Browser Webcam
     │  (JavaScript captures frames)
     ▼
Base64 Frame → Python Kernel
     │
     ▼
┌─────────────────────────────────┐
│   SleepGuard ADK Agent System  │
│                                 │
│  ┌─────────────────────────┐   │
│  │  CoordinatorAgent       │   │  ← routes frames
│  │  (google/adk LlmAgent)  │   │
│  └──────────┬──────────────┘   │
│             │ delegates to     │
│  ┌──────────▼──────────────┐   │
│  │  VisionAnalyzerAgent    │   │  ← Gemini 2.0 Flash sees frame
│  │  (Gemini Flash Vision)  │   │    returns JSON: EAR,MAR,pose,phone
│  └──────────┬──────────────┘   │
│             │ results to       │
│  ┌──────────▼──────────────┐   │
│  │  AlertAgent             │   │  ← scores attention, triggers alerts
│  │  (rule-based + Gemini)  │   │
│  └─────────────────────────┘   │
└─────────────────────────────────┘
     │
     ▼
HUD Overlay → Display in Notebook
```

### ✅ Works on: Kaggle · Google Colab · Local Jupyter

</div>

---
## 📦 CELL 1 — Install All Dependencies
> **Run this first.** Installs Google ADK, MediaPipe (fallback), OpenCV, and all required libs.

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 1 — Install Dependencies
#  Includes: Google ADK, MediaPipe, OpenCV, Gemini SDK
# ════════════════════════════════════════════════════════════
import subprocess, sys

PACKAGES = [
    ("google-adk",                 "Google Agent Development Kit"),
    ("google-generativeai",        "Gemini Vision API"),
    ("opencv-python-headless",     "OpenCV (headless for Kaggle/Colab)"),
    ("mediapipe",                  "MediaPipe face/hand detection"),
    ("scipy",                      "EAR/MAR distance math"),
    ("numpy",                      "Array operations"),
    ("Pillow",                     "Image encode/decode"),
    ("asyncio",                    "Async runtime"),
    ("nest-asyncio",               "Allow asyncio in Jupyter"),
]

print("📦 Installing packages...\n")
for pkg, desc in PACKAGES:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if r.returncode == 0 else "⚠️"
    print(f"  {status}  {pkg:35s}  # {desc}")

print("\n🎉 All packages ready! Proceed to Cell 2.")

---
## 🔑 CELL 2 — API Key Setup
> Set your **Gemini API key** here. Get one free at https://aistudio.google.com/apikey
>
> On Kaggle: Add it as a **Secret** named `GEMINI_API_KEY` → it auto-loads below.
>
> On Colab: Same — use the 🔑 Secrets panel.

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 2 — API Key Configuration
#  Supports: Kaggle Secrets / Colab Secrets / Manual entry
# ════════════════════════════════════════════════════════════
import os

GEMINI_API_KEY = ""

# ── Auto-load from Kaggle Secrets ─────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("✅ Loaded from Kaggle Secrets")
except Exception:
    pass

# ── Auto-load from Colab Secrets ──────────────────────────
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
        print("✅ Loaded from Colab Secrets")
    except Exception:
        pass

# ── Auto-load from environment variable ───────────────────
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    if GEMINI_API_KEY:
        print("✅ Loaded from environment variable")

# ── MANUAL FALLBACK — paste your key here ─────────────────
if not GEMINI_API_KEY:
    GEMINI_API_KEY = ""  # ← PASTE YOUR KEY HERE if not using secrets
    print("⚠️  Using manual key — consider using Kaggle Secrets instead")

if not GEMINI_API_KEY:
    print("❌ No API key found!")
    print("   Get one free: https://aistudio.google.com/apikey")
    print("   Then add as Kaggle Secret named 'GEMINI_API_KEY'")
else:
    masked = GEMINI_API_KEY[:8] + "*" * (len(GEMINI_API_KEY) - 12) + GEMINI_API_KEY[-4:]
    print(f"\n🔑 API Key: {masked}")
    print("✅ Ready for Gemini Vision!")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

---
## ⚙️ CELL 3 — Core Imports & Configuration
> All detection thresholds, scoring weights, and colors live here.
> **Tune any value freely** without touching other cells.

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 3 — Imports & Global Configuration
# ════════════════════════════════════════════════════════════
import cv2, mediapipe as mp, numpy as np
from scipy.spatial import distance as dist
from collections import deque
import base64, time, warnings, io, json, asyncio, re, textwrap
from IPython.display import display, Javascript, HTML, clear_output
from PIL import Image as PILImage
import google.generativeai as genai
import nest_asyncio
nest_asyncio.apply()          # ← makes asyncio work inside Jupyter
warnings.filterwarnings("ignore")

# ── Configure Gemini ──────────────────────────────────────
genai.configure(api_key=GEMINI_API_KEY)

# ════════════════════════════════════════════════════════════
#  DETECTION THRESHOLDS  (tune these freely)
# ════════════════════════════════════════════════════════════
CFG = {
    # ── Eye Aspect Ratio ──────────────────────────────────
    # Normal open eye ≈ 0.28-0.35 │ Closed < 0.22
    "EAR_THRESH":    0.22,
    "EAR_CONSEC":    15,    # consecutive closed frames → SLEEPING

    # ── Mouth Aspect Ratio ────────────────────────────────
    # Normal closed ≈ 0.30 │ Yawning > 0.60
    "MAR_THRESH":    0.60,
    "YAWN_CONSEC":   18,    # consecutive yawn frames → count yawn

    # ── Head Pose (degrees) ───────────────────────────────
    "PITCH_T":       15,    # nodding forward/down
    "YAW_T":         30,    # looking sideways
    "ROLL_T":        20,    # tilting (sleeping on shoulder)

    # ── Attention Score Weights (sum = 1.0) ───────────────
    "W_EYE":         0.40,
    "W_HEAD":        0.30,
    "W_YAWN":        0.15,
    "W_PHONE":       0.15,

    # ── ADK Agent frame rate ──────────────────────────────
    # Gemini Vision called every N frames (reduces API cost)
    "AGENT_EVERY":   10,    # call Gemini every 10 frames

    # ── Display ───────────────────────────────────────────
    "MAX_FRAMES":    500,   # stop after N frames (0 = infinite)
    "FRAME_W":       640,
    "FRAME_H":       480,
}

# MediaPipe landmark indices
L_EYE  = [362, 385, 387, 263, 373, 380]
R_EYE  = [33,  160, 158, 133, 153, 144]
L_RING = [362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398]
R_RING = [33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246]
MOUTH  = [61, 291, 39, 181, 0, 17, 269, 405]
POSE_I = [1, 152, 226, 446, 57, 287]
POSE_3D = np.array([
    (  0.0,    0.0,    0.0),
    (  0.0, -330.0,  -65.0),
    (-225.0,  170.0, -135.0),
    ( 225.0,  170.0, -135.0),
    (-150.0, -150.0, -125.0),
    ( 150.0, -150.0, -125.0),
], dtype=np.float64)

# Colors (BGR for OpenCV)
C = {
    "ok":    ( 70, 220,  70),
    "warn":  (  0, 160, 255),
    "alert": ( 30,  30, 240),
    "phone": (180,   0, 200),
    "cyan":  (200, 220,   0),
    "white": (255, 255, 255),
    "dim":   ( 90, 100, 120),
    "panel": ( 18,  24,  40),
    "info":  (  0, 200, 200),
}
FONT = cv2.FONT_HERSHEY_SIMPLEX

print("✅ Config loaded!")
print(f"   EAR threshold   : {CFG['EAR_THRESH']} (< = eye closed)")
print(f"   MAR threshold   : {CFG['MAR_THRESH']} (> = yawning)")
print(f"   Agent call rate : every {CFG['AGENT_EVERY']} frames")
print(f"   Gemini model    : gemini-2.0-flash (vision capable)")

---
## 🤖 CELL 4 — Google ADK Agent System
> This is the **brain** of SleepGuard.
>
> **3 specialized agents** work together:
> - `VisionAnalyzerAgent` — sends webcam frame to Gemini Flash, gets back structured JSON with eye/head/yawn/phone analysis
> - `AlertAgent` — computes attention score, sleep probability, determines alert level
> - `CoordinatorAgent` — orchestrates both sub-agents, maintains session state

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 4 — Google ADK Multi-Agent System
#  Architecture: Coordinator → VisionAnalyzer + AlertAgent
# ════════════════════════════════════════════════════════════
from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part
import google.genai as gai


# ────────────────────────────────────────────────────────────
#  TOOL 1: Gemini Vision Frame Analyzer
#  Sends a JPEG frame to Gemini, returns structured JSON
# ────────────────────────────────────────────────────────────
def analyze_frame_with_vision(frame_b64: str) -> str:
    """
    Analyze a webcam frame for drowsiness indicators.
    Sends the frame to Gemini Vision and returns a JSON string
    with eye_open (bool), yawning (bool), head_down (bool),
    phone_visible (bool), and a brief observation string.

    Args:
        frame_b64: Base64-encoded JPEG frame from webcam

    Returns:
        JSON string with detection results
    """
    try:
        client = gai.Client(api_key=GEMINI_API_KEY)

        prompt = textwrap.dedent("""
            You are a classroom attention monitoring system.
            Analyze this webcam frame of a student and return ONLY valid JSON.

            Detect these indicators:
            1. eyes_open: Are both eyes clearly open? (true/false)
            2. yawning: Is the person yawning (mouth wide open)? (true/false)
            3. head_down: Is the head tilted down or drooping? (true/false)
            4. head_sideways: Is the head turned far sideways? (true/false)
            5. phone_visible: Is a phone visible or being held? (true/false)
            6. face_visible: Is a face clearly visible in frame? (true/false)
            7. observation: One short sentence describing the student state.

            Return ONLY this JSON, no other text:
            {
              "eyes_open": true,
              "yawning": false,
              "head_down": false,
              "head_sideways": false,
              "phone_visible": false,
              "face_visible": true,
              "observation": "Student appears alert and attentive."
            }
        """)

        # Decode base64 → bytes for Gemini
        img_bytes = base64.b64decode(frame_b64)

        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=[
                {"role": "user", "parts": [
                    {"inline_data": {"mime_type": "image/jpeg", "data": frame_b64}},
                    {"text": prompt}
                ]}
            ]
        )

        raw = response.text.strip()
        # Strip markdown code fences if present
        raw = re.sub(r"```json\s*|```\s*", "", raw).strip()
        return raw

    except Exception as e:
        return json.dumps({
            "eyes_open": True, "yawning": False,
            "head_down": False, "head_sideways": False,
            "phone_visible": False, "face_visible": False,
            "observation": f"Vision error: {str(e)[:60]}"
        })


# ────────────────────────────────────────────────────────────
#  TOOL 2: Attention Score Calculator
#  Pure rule-based scoring from vision JSON output
# ────────────────────────────────────────────────────────────
def compute_attention_score(
    eyes_open: bool,
    yawning: bool,
    head_down: bool,
    head_sideways: bool,
    phone_visible: bool,
    consecutive_closed_frames: int = 0,
    consecutive_yawn_frames: int = 0,
) -> str:
    """
    Compute attention score and sleep probability.
    Returns JSON with attention_score (0-100), sleep_probability (0-100),
    alert_level (ATTENTIVE/DROWSY/DISTRACTED/SLEEPING), and badges list.

    Args:
        eyes_open: Whether eyes are open
        yawning: Whether person is yawning
        head_down: Whether head is tilted down
        head_sideways: Whether head is turned sideways
        phone_visible: Whether phone is visible
        consecutive_closed_frames: Frames eyes have been consecutively closed
        consecutive_yawn_frames: Frames mouth has been consecutively open
    """
    # Individual distraction scores [0..1]
    eye_score  = 0.0 if eyes_open else 1.0
    head_score = min((int(head_down) + int(head_sideways)) / 2.0, 1.0)
    yawn_score = 1.0 if yawning else 0.0
    phone_score= 1.0 if phone_visible else 0.0

    # Weighted distraction (0=fully attentive, 1=fully distracted)
    distraction = (
        CFG["W_EYE"]   * eye_score   +
        CFG["W_HEAD"]  * head_score  +
        CFG["W_YAWN"]  * yawn_score  +
        CFG["W_PHONE"] * phone_score
    )
    attention = round(100.0 * (1.0 - distraction), 1)

    # Sleep probability (eye+head+yawn only, no phone)
    sleep_prob = round(100.0 * np.clip(
        0.55 * eye_score + 0.30 * head_score + 0.15 * yawn_score, 0, 1
    ), 1)

    # Boost sleep prob if eyes consecutively closed
    if consecutive_closed_frames >= CFG["EAR_CONSEC"]:
        sleep_prob = min(sleep_prob + 30, 100)

    # Alert level
    if sleep_prob > 70 or consecutive_closed_frames >= CFG["EAR_CONSEC"]:
        level = "SLEEPING"
    elif distraction > 0.55:
        level = "DISTRACTED"
    elif distraction > 0.30:
        level = "DROWSY"
    else:
        level = "ATTENTIVE"

    # Alert badges
    badges = []
    if consecutive_closed_frames >= CFG["EAR_CONSEC"]: badges.append("SLEEPING")
    if yawning:        badges.append("YAWNING")
    if head_down:      badges.append("HEAD DOWN")
    if head_sideways:  badges.append("LOOKING AWAY")
    if phone_visible:  badges.append("PHONE USE")

    return json.dumps({
        "attention_score":  attention,
        "sleep_probability": sleep_prob,
        "alert_level":      level,
        "badges":           badges,
        "eye_score":        round(eye_score * 100),
        "head_score":       round(head_score * 100),
        "yawn_score":       round(yawn_score * 100),
        "phone_score":      round(phone_score * 100),
    })


# ────────────────────────────────────────────────────────────
#  ADK AGENTS
# ────────────────────────────────────────────────────────────

# Agent 1: Specialized vision analysis
vision_agent = LlmAgent(
    name="VisionAnalyzerAgent",
    model="gemini-2.0-flash",
    description="Analyzes webcam frames for drowsiness indicators using Gemini Vision.",
    instruction="""
        You are a vision analysis specialist.
        When given a base64 frame, use the analyze_frame_with_vision tool to analyze it.
        Return the raw JSON result from the tool.
        Do not add any extra commentary.
    """,
    tools=[analyze_frame_with_vision],
)

# Agent 2: Alert decision maker
alert_agent = LlmAgent(
    name="AlertAgent",
    model="gemini-2.0-flash",
    description="Computes attention score and determines alert level from vision data.",
    instruction="""
        You are an attention scoring specialist.
        When given vision analysis JSON, call compute_attention_score with the boolean fields.
        Return the raw JSON result.
        Do not add any extra commentary.
    """,
    tools=[compute_attention_score],
)

# Agent 3: Coordinator orchestrates both
coordinator_agent = LlmAgent(
    name="CoordinatorAgent",
    model="gemini-2.0-flash",
    description="Orchestrates the SleepGuard detection pipeline.",
    instruction="""
        You are the SleepGuard AI coordinator.
        Your job: when given a base64 JPEG frame, run this 2-step pipeline:

        Step 1: Delegate to VisionAnalyzerAgent to analyze the frame.
        Step 2: Delegate to AlertAgent to compute scores from the vision results.

        Return ONLY the final JSON from AlertAgent merged with vision observation.
        Format:
        {
          "attention_score": <number>,
          "sleep_probability": <number>,
          "alert_level": <string>,
          "badges": [<strings>],
          "observation": <string>,
          "eyes_open": <bool>,
          "yawning": <bool>,
          "head_down": <bool>,
          "phone_visible": <bool>
        }
    """,
    sub_agents=[vision_agent, alert_agent],
)

print("✅ ADK Agent System initialized!")
print("   📡 VisionAnalyzerAgent  → Gemini 2.0 Flash Vision")
print("   📊 AlertAgent           → Attention scoring + alert logic")
print("   🎯 CoordinatorAgent     → Orchestrates full pipeline")

---
## 🧮 CELL 5 — MediaPipe Local Detection Engine
> **Fast local detection** using MediaPipe runs on EVERY frame.
>
> Gemini Vision (ADK Agent) runs every N frames as a **higher-level verifier**.
>
> This hybrid approach gives you **real-time speed + AI intelligence**.

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 5 — MediaPipe Detection Engine (runs every frame)
# ════════════════════════════════════════════════════════════

def lm_pt(lm, i, W, H):
    p = lm[i]; return (p.x * W, p.y * H)

def lm_px(lm, i, W, H):
    x, y = lm_pt(lm, i, W, H); return int(x), int(y)

def calc_EAR(lm, eye6, W, H):
    """
    EAR = (||p1-p5|| + ||p2-p4||) / (2 * ||p0-p3||)
    Open ≈ 0.30 │ Closed < 0.22
    """
    p  = [lm_pt(lm, i, W, H) for i in eye6]
    A  = dist.euclidean(p[1], p[5])
    B  = dist.euclidean(p[2], p[4])
    CC = dist.euclidean(p[0], p[3])
    return (A + B) / (2.0 * CC + 1e-7)

def calc_MAR(lm, W, H):
    """
    MAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p0-p4||)
    Closed ≈ 0.30 │ Yawning > 0.60
    """
    p  = [lm_pt(lm, i, W, H) for i in MOUTH]
    A  = dist.euclidean(p[2], p[6])
    B  = dist.euclidean(p[3], p[5])
    CC = dist.euclidean(p[0], p[4])
    return (A + B) / (2.0 * CC + 1e-7)

def calc_pose(lm, W, H):
    """
    Returns (pitch°, yaw°, roll°) via OpenCV solvePnP.
    pitch > PITCH_T  → nodding (sleepy)
    roll  > ROLL_T   → head tilt (sleeping on shoulder)
    """
    img_pts = np.array([lm_pt(lm, i, W, H) for i in POSE_I], dtype=np.float64)
    fl = W
    cam = np.array([[fl,0,W/2],[0,fl,H/2],[0,0,1]], dtype=np.float64)
    ok, rvec, tvec = cv2.solvePnP(POSE_3D, img_pts, cam, np.zeros((4,1)),
                                   flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok: return 0.0, 0.0, 0.0
    rmat, _ = cv2.Rodrigues(rvec)
    pm = cv2.hconcat([rmat, tvec])
    ext = cv2.hconcat([pm, np.array([[0],[0],[0],[1]], dtype=np.float64).T])
    _, _, _, _, _, _, ea = cv2.decomposeProjectionMatrix(ext)
    return float(ea[0][0]), float(ea[1][0]), float(ea[2][0])


# ── Session State (shared across all frames) ──────────────
class SessionState:
    """Persistent state across frames."""
    def __init__(self):
        self.t0             = time.time()
        self.frame_num      = 0
        self.eye_cf         = 0     # consecutive closed frames
        self.yawn_cf        = 0     # consecutive yawn frames
        self.blinks         = 0
        self.yawns          = 0
        self.sleep_eps      = 0
        self.phone_eps      = 0
        self.sleep_secs     = 0.0
        self._sleep_start   = None
        self._prev_sleep    = False
        self._prev_phone    = False
        # Latest values
        self.EAR            = 0.30
        self.MAR            = 0.30
        self.pitch          = 0.0
        self.yaw            = 0.0
        self.roll           = 0.0
        self.attention      = 100.0
        self.sleep_prob     = 0.0
        self.alert_level    = "ATTENTIVE"
        self.badges         = []
        self.observation    = "Initializing..."
        self.agent_result   = {}
        self.fps            = 0.0
        self._fps_t         = time.time()
        self._fps_cnt       = 0

    def elapsed(self):
        e = int(time.time() - self.t0)
        return f"{e//60:02d}:{e%60:02d}"

    def tick_fps(self):
        self._fps_cnt += 1
        if time.time() - self._fps_t >= 1.0:
            self.fps = self._fps_cnt
            self._fps_cnt = 0
            self._fps_t = time.time()

    def update_events(self, is_sleeping, is_phone):
        t = time.time()
        if is_sleeping and not self._prev_sleep:
            self.sleep_eps += 1
            self._sleep_start = t
        if not is_sleeping and self._prev_sleep and self._sleep_start:
            self.sleep_secs += t - self._sleep_start
            self._sleep_start = None
        if is_phone and not self._prev_phone:
            self.phone_eps += 1
        self._prev_sleep = is_sleeping
        self._prev_phone = is_phone


SESSION = SessionState()

print("✅ MediaPipe Detection Engine ready")
print("✅ Session State initialized")
print("   Running: MediaPipe every frame + Gemini every",
      CFG['AGENT_EVERY'], "frames")

---
## 🖥️ CELL 6 — HUD Overlay Renderer
> Draws the full heads-up display on every frame:
> metric bars, circular gauges, alert badges, face bounding box, status bar.

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 6 — HUD Overlay Renderer
# ════════════════════════════════════════════════════════════

def alpha_rect(img, x1, y1, x2, y2, color, alpha=0.75, r=6):
    """Draw a semi-transparent rounded rectangle."""
    ov = img.copy()
    if r > 0:
        cv2.rectangle(ov, (x1+r,y1), (x2-r,y2), color, -1)
        cv2.rectangle(ov, (x1,y1+r), (x2,y2-r), color, -1)
        for cx,cy in [(x1+r,y1+r),(x2-r,y1+r),(x1+r,y2-r),(x2-r,y2-r)]:
            cv2.circle(ov, (cx,cy), r, color, -1)
    else:
        cv2.rectangle(ov, (x1,y1), (x2,y2), color, -1)
    cv2.addWeighted(ov, alpha, img, 1-alpha, 0, img)

def draw_bar(img, x, y, w, h, val, maxv, label="", thresh=None,
             lo=(0,220,80), hi=(30,30,240)):
    """Horizontal gradient progress bar with optional threshold line."""
    cv2.rectangle(img, (x,y), (x+w,y+h), (40,45,60), -1)
    ratio = float(np.clip(val / max(maxv, 1e-6), 0, 1))
    fw = int(w * ratio)
    if fw > 0:
        br = int(lo[2]*(1-ratio) + hi[2]*ratio)
        bg = int(lo[1]*(1-ratio) + hi[1]*ratio)
        bb = int(lo[0]*(1-ratio) + hi[0]*ratio)
        cv2.rectangle(img, (x,y), (x+fw,y+h), (bb,bg,br), -1)
    cv2.rectangle(img, (x,y), (x+w,y+h), (60,65,80), 1)
    if thresh is not None:
        tx = x + int(w * thresh / max(maxv, 1e-6))
        if x < tx < x+w:
            cv2.line(img, (tx,y), (tx,y+h), (0,220,220), 2)
    if label:
        cv2.putText(img, label, (x,y-3), FONT, 0.35, C["dim"], 1, cv2.LINE_AA)

def draw_gauge(img, cx, cy, radius, val, maxv, color, label=""):
    """Circular arc gauge."""
    cv2.ellipse(img, (cx,cy), (radius,radius), -90, 0, 270, (45,50,65), 5)
    deg = int(270 * np.clip(val/max(maxv,1e-6), 0, 1))
    if deg > 0:
        cv2.ellipse(img, (cx,cy), (radius,radius), -90, 0, deg, color, 5)
    vs = f"{val:.0f}%"
    ts = cv2.getTextSize(vs, FONT, 0.52, 1)[0]
    cv2.putText(img, vs, (cx-ts[0]//2, cy+7), FONT, 0.52, C["white"], 1, cv2.LINE_AA)
    ls = cv2.getTextSize(label, FONT, 0.34, 1)[0]
    cv2.putText(img, label, (cx-ls[0]//2, cy+radius+15), FONT, 0.34, C["dim"], 1, cv2.LINE_AA)

def corner_box(img, x, y, w, h, color, t=2):
    """Corner accent marks for face bounding box."""
    s = 16
    for px,py,dx,dy in [(x,y,1,1),(x+w,y,-1,1),(x,y+h,1,-1),(x+w,y+h,-1,-1)]:
        cv2.line(img, (px,py), (px+dx*s,py), color, t)
        cv2.line(img, (px,py), (px,py+dy*s), color, t)


# ── Alert level → display color ───────────────────────────
LEVEL_COLOR = {
    "ATTENTIVE":  C["ok"],
    "DROWSY":     C["warn"],
    "DISTRACTED": (0, 140, 255),
    "SLEEPING":   C["alert"],
}


def render_hud(frame, s: SessionState, face_bbox=None):
    """
    Render the complete HUD overlay on `frame`.
    `s` is the SessionState holding all current metrics.
    """
    H, W = frame.shape[:2]
    level  = s.alert_level
    lcolor = LEVEL_COLOR.get(level, C["ok"])

    # ── Top Banner ────────────────────────────────────────
    bc = tuple(max(c//4,0) for c in lcolor)
    alpha_rect(frame, 0,0, W,40, bc, 0.88, 0)
    cv2.line(frame, (0,40), (W,40), lcolor, 2)
    cv2.putText(frame, f"SleepGuard AI  |  {level}  |  FPS:{s.fps:.0f}  |  {s.elapsed()}",
                (10,27), FONT, 0.58, C["white"], 1, cv2.LINE_AA)
    # ADK agent badge top-right
    adk_txt = "ADK+Gemini"
    atsz = cv2.getTextSize(adk_txt, FONT, 0.38, 1)[0]
    cv2.putText(frame, adk_txt, (W-atsz[0]-8, 15), FONT, 0.38, C["cyan"], 1, cv2.LINE_AA)
    cv2.putText(frame, "AI Powered", (W-atsz[0]-5, 30), FONT, 0.34, C["dim"], 1, cv2.LINE_AA)

    # ── Left Panel — Metrics ──────────────────────────────
    LX, LY, LW = 6, 48, 200
    alpha_rect(frame, LX,LY, LX+LW,LY+330, C["panel"], 0.82)

    y = LY + 16
    cv2.putText(frame, "MEDIAPIPE METRICS", (LX+6,y), FONT, 0.38, C["cyan"], 1, cv2.LINE_AA)
    cv2.line(frame, (LX+4,y+4), (LX+LW-4,y+4), (40,60,80), 1)

    y += 18
    draw_bar(frame, LX+4,y, LW-10,10, s.EAR, 0.45,
             f"EAR  {s.EAR:.3f}", CFG["EAR_THRESH"],
             (0,200,60),(30,30,220))
    y += 28
    draw_bar(frame, LX+4,y, LW-10,10, s.MAR, 1.2,
             f"MAR  {s.MAR:.3f}", CFG["MAR_THRESH"],
             (0,200,60),(30,30,220))

    y += 28
    cv2.putText(frame, "HEAD POSE", (LX+6,y), FONT, 0.34, C["dim"], 1, cv2.LINE_AA)
    y += 12
    for lbl, val, lim in [
        (f"Pitch {s.pitch:+.1f}°", abs(s.pitch), CFG["PITCH_T"]),
        (f"Yaw   {s.yaw:+.1f}°",   abs(s.yaw),   CFG["YAW_T"]),
        (f"Roll  {s.roll:+.1f}°",   abs(s.roll),   CFG["ROLL_T"]),
    ]:
        draw_bar(frame, LX+4,y, LW-10,9, val, lim*1.5, lbl, thresh=lim,
                 lo=(0,180,60), hi=(30,30,220))
        y += 25

    y += 4
    cv2.putText(frame, "SESSION EVENTS", (LX+6,y), FONT, 0.34, C["dim"], 1, cv2.LINE_AA)
    y += 14
    for lbl, val, col in [
        (f"Blinks       {s.blinks}",    s.blinks,      C["info"]),
        (f"Yawns        {s.yawns}",     s.yawns,       C["warn"]),
        (f"Sleep Eps    {s.sleep_eps}", s.sleep_eps,   C["alert"]),
        (f"Phone Eps    {s.phone_eps}", s.phone_eps,   C["phone"]),
        (f"Sleep Time   {s.sleep_secs:.1f}s", None,   C["alert"]),
    ]:
        cv2.putText(frame, lbl, (LX+6,y), FONT, 0.38, col, 1, cv2.LINE_AA)
        y += 16

    # ── Right Panel — Gauges ──────────────────────────────
    RW = 155; RX = W - RW - 6
    alpha_rect(frame, RX,48, RX+RW,48+310, C["panel"], 0.82)

    cv2.putText(frame, "AI SCORES", (RX+8,66), FONT, 0.38, C["cyan"], 1, cv2.LINE_AA)
    cv2.line(frame, (RX+4,70), (RX+RW-4,70), (40,60,80), 1)

    ac = C["alert"] if s.attention<40 else C["warn"] if s.attention<70 else C["ok"]
    draw_gauge(frame, RX+RW//2, 128, 48, s.attention, 100, ac, "ATTENTION")

    sc = C["alert"] if s.sleep_prob>70 else C["warn"] if s.sleep_prob>35 else C["ok"]
    draw_gauge(frame, RX+RW//2, 250, 48, s.sleep_prob, 100, sc, "SLEEP PROB")

    # Gemini observation text
    obs = s.observation[:55] if s.observation else ""
    cv2.putText(frame, "Gemini says:", (RX+4, 310), FONT, 0.33, C["cyan"], 1, cv2.LINE_AA)
    # Word wrap to 2 lines
    words = obs.split()
    line1, line2 = "", ""
    for w in words:
        if len(line1) + len(w) < 22: line1 += w + " "
        else: line2 += w + " "
    cv2.putText(frame, line1.strip(), (RX+4, 322), FONT, 0.30, C["white"], 1, cv2.LINE_AA)
    cv2.putText(frame, line2.strip()[:26], (RX+4, 334), FONT, 0.30, C["dim"], 1, cv2.LINE_AA)

    # Frame counter
    cv2.putText(frame, f"Frame #{s.frame_num}", (RX+4, 348), FONT, 0.32, C["dim"], 1, cv2.LINE_AA)

    # ── Alert Badges ──────────────────────────────────────
    if s.badges:
        badge_colors = {
            "SLEEPING":     C["alert"],
            "YAWNING":      C["warn"],
            "HEAD DOWN":    C["warn"],
            "LOOKING AWAY": (0,130,220),
            "PHONE USE":    C["phone"],
        }
        bh = 28
        total_w = sum(len(b)*9+20 for b in s.badges) + (len(s.badges)-1)*6
        bx = max((W - total_w) // 2, 5)
        by = H - 50
        for badge in s.badges:
            bcol = badge_colors.get(badge, C["warn"])
            bw   = len(badge)*9 + 20
            alpha_rect(frame, bx,by, bx+bw,by+bh, bcol, 0.88, 5)
            ts = cv2.getTextSize(badge, FONT, 0.48, 2)[0]
            cv2.putText(frame, badge, (bx+(bw-ts[0])//2, by+19),
                        FONT, 0.48, C["white"], 2, cv2.LINE_AA)
            bx += bw + 6

    # ── Face Box ──────────────────────────────────────────
    if face_bbox:
        fx,fy,fw,fh = face_bbox
        cv2.rectangle(frame, (fx,fy), (fx+fw,fy+fh), lcolor, 1)
        corner_box(frame, fx,fy,fw,fh, lcolor, 3)

    # ── Bottom Status Bar ─────────────────────────────────
    alpha_rect(frame, 0,H-24, W,H, (10,14,24), 0.88, 0)
    cv2.line(frame, (0,H-24), (W,H-24), (30,45,65), 1)
    status = (f"  EAR:{s.EAR:.3f}  MAR:{s.MAR:.3f}  "
              f"P:{s.pitch:+.1f} Y:{s.yaw:+.1f} R:{s.roll:+.1f}  |  "
              f"Att:{s.attention:.0f}%  Sleep:{s.sleep_prob:.0f}%  |  "
              f"Blinks:{s.blinks}  Yawns:{s.yawns}  SleepEps:{s.sleep_eps}")
    cv2.putText(frame, status, (4,H-8), FONT, 0.32, C["dim"], 1, cv2.LINE_AA)

    return frame


print("✅ HUD Renderer ready")
print("   Draws: EAR/MAR bars, head pose bars, circular gauges,")
print("          alert badges, face bbox, Gemini observation, status bar")

---
## 🌐 CELL 7 — JavaScript Webcam Bridge
> This cell builds the JavaScript code that runs in your **browser**
> and connects your webcam to the Python kernel.
>
> **How it works:**
> 1. JS creates a `<video>` element connected to `getUserMedia()`
> 2. Every frame, JS draws video onto a `<canvas>` and encodes as base64 JPEG
> 3. base64 string is passed to Python via `google.colab.kernel.invokeFunction` (Colab) or `IPython.notebook.kernel.execute` (Kaggle)

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 7 — JavaScript Webcam Bridge + Frame Capture Loop
# ════════════════════════════════════════════════════════════

WEBCAM_JS = """
(async () => {
  // ── Cleanup previous session ──────────────────────────
  if (window._sg_stream) {
    window._sg_stream.getTracks().forEach(t => t.stop());
    window._sg_active = false;
  }
  if (window._sg_interval) clearInterval(window._sg_interval);

  // ── Create UI elements ────────────────────────────────
  const div = document.getElementById('sg_display') || document.createElement('div');
  div.id = 'sg_display';
  div.style.cssText = `
    background:#0a0f1e; padding:12px; border-radius:10px;
    border:1px solid #1e3a5f; margin:8px 0; font-family:monospace;
  `;
  div.innerHTML = `
    <div style="color:#00d4ff;font-size:15px;font-weight:bold;margin-bottom:8px">
      🛡️ SleepGuard AI — Requesting webcam...</div>
    <video id="sg_video" width="320" height="240"
           style="border-radius:6px;display:none"></video>
    <canvas id="sg_canvas" width="640" height="480"
           style="display:none"></canvas>
    <div id="sg_status" style="color:#aaa;font-size:12px;margin-top:6px">
      Waiting for camera permission...</div>
    <div id="sg_stats" style="color:#00ff88;font-size:11px;margin-top:4px"></div>
  `;
  document.querySelector('#output, .output, [data-mime-type]')?.prepend(div)
    || document.body.prepend(div);

  const video  = document.getElementById('sg_video');
  const canvas = document.getElementById('sg_canvas');
  const ctx    = canvas.getContext('2d');
  const status = document.getElementById('sg_status');
  const stats  = document.getElementById('sg_stats');

  // ── Get webcam stream ─────────────────────────────────
  let stream;
  try {
    stream = await navigator.mediaDevices.getUserMedia({
      video: { width: 640, height: 480, facingMode: 'user' },
      audio: false
    });
  } catch(e) {
    status.textContent = '❌ Camera error: ' + e.message;
    status.style.color = '#ff4444';
    return;
  }

  window._sg_stream = stream;
  window._sg_active = true;
  video.srcObject = stream;
  await video.play();
  status.textContent = '✅ Camera active — processing frames...';
  status.style.color = '#00ff88';

  let frameCount = 0;
  let sentCount  = 0;

  // ── Frame capture loop ────────────────────────────────
  window._sg_interval = setInterval(() => {
    if (!window._sg_active) {
      clearInterval(window._sg_interval);
      return;
    }
    if (video.readyState !== 4) return;  // frame not ready yet

    frameCount++;
    ctx.drawImage(video, 0, 0, 640, 480);
    const b64 = canvas.toDataURL('image/jpeg', 0.80).split(',')[1];

    // Send frame to Python kernel
    // Works on both Kaggle and Colab
    try {
      // Colab method
      google.colab.kernel.invokeFunction('notebook.process_frame', [b64], {});
    } catch {
      // Kaggle / Jupyter method
      const cmd = `_SG_FRAME_BUFFER = '${b64}'`;
      IPython.notebook.kernel.execute(cmd);
    }

    sentCount++;
    stats.textContent = `Frames captured: ${frameCount} | Sent to Python: ${sentCount}`;

  }, 100);  // 10 FPS capture rate (balance speed vs API cost)

  console.log('SleepGuard AI webcam bridge active');
})();
"""

print("✅ JavaScript webcam bridge code prepared")
print("   Capture rate  : 10 FPS")
print("   Resolution    : 640×480")
print("   Format        : JPEG @ 80% quality")
print("   Compatible    : Kaggle + Colab + Local Jupyter")

---
## 🔄 CELL 8 — Frame Processor Pipeline
> The core processing loop. For each frame:
> 1. Decode base64 → OpenCV image
> 2. Run MediaPipe FaceMesh + Hands (local, fast, every frame)
> 3. Every N frames → call ADK Coordinator Agent (Gemini Vision)
> 4. Merge results → update SessionState → render HUD → display

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 8 — Frame Processor: MediaPipe + ADK Agent Pipeline
# ════════════════════════════════════════════════════════════

# ── Initialize MediaPipe ──────────────────────────────────
_mp_fm = mp.solutions.face_mesh
_mp_hd = mp.solutions.hands

FACE_MESH = _mp_fm.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
HANDS = _mp_hd.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.55,
    min_tracking_confidence=0.5,
)
MP_DRAW = mp.solutions.drawing_utils


def frame_from_b64(b64_str: str):
    """Decode base64 JPEG string → OpenCV BGR numpy array."""
    jpg = base64.b64decode(b64_str)
    arr = np.frombuffer(jpg, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


def frame_to_b64(frame) -> str:
    """OpenCV BGR frame → base64 JPEG string for display."""
    _, jpg = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 82])
    return base64.b64encode(jpg).decode("utf-8")


def run_mediapipe(frame, s: SessionState):
    """
    Run MediaPipe face mesh + hands on one frame.
    Updates SessionState with EAR, MAR, pose.
    Returns (face_bbox, face_landmarks, hand_detected).
    """
    H, W = frame.shape[:2]
    rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    face_bbox = None
    face_lm   = None
    hand_detected = False

    # ── Face Mesh ──────────────────────────────────────
    fm_res = FACE_MESH.process(rgb)
    if fm_res.multi_face_landmarks:
        lm = fm_res.multi_face_landmarks[0].landmark
        face_lm = lm

        # EAR
        l_ear = calc_EAR(lm, L_EYE, W, H)
        r_ear = calc_EAR(lm, R_EYE, W, H)
        s.EAR = (l_ear + r_ear) / 2.0

        # MAR
        s.MAR = calc_MAR(lm, W, H)

        # Head Pose
        try:
            s.pitch, s.yaw, s.roll = calc_pose(lm, W, H)
        except Exception:
            pass

        # Eye consecutive frames
        if s.EAR < CFG["EAR_THRESH"]:
            s.eye_cf += 1
        else:
            if s.eye_cf >= 2: s.blinks += 1
            s.eye_cf = 0

        # Yawn consecutive frames
        if s.MAR > CFG["MAR_THRESH"]:
            s.yawn_cf += 1
        else:
            if s.yawn_cf >= CFG["YAWN_CONSEC"]: s.yawns += 1
            s.yawn_cf = 0

        # Face bounding box from landmarks
        xs = [int(lm[i].x * W) for i in range(468)]
        ys = [int(lm[i].y * H) for i in range(468)]
        x1,y1 = max(min(xs)-8,0), max(min(ys)-8,0)
        x2,y2 = min(max(xs)+8,W), min(max(ys)+8,H)
        face_bbox = (x1, y1, x2-x1, y2-y1)

        # Draw eye contours
        ec = C["alert"] if s.EAR < CFG["EAR_THRESH"] else C["ok"]
        for ring in [L_RING, R_RING]:
            pts = np.array([[int(lm[i].x*W), int(lm[i].y*H)] for i in ring], np.int32)
            cv2.polylines(frame, [pts], True, ec, 1, cv2.LINE_AA)

        # Draw mouth outline
        mc = C["alert"] if s.MAR > CFG["MAR_THRESH"] else C["dim"]
        mpts = np.array([[int(lm[i].x*W), int(lm[i].y*H)] for i in MOUTH], np.int32)
        cv2.polylines(frame, [mpts], True, mc, 1, cv2.LINE_AA)

    # ── Hands ──────────────────────────────────────────
    hd_res = HANDS.process(rgb)
    if hd_res.multi_hand_landmarks:
        for hlm in hd_res.multi_hand_landmarks:
            MP_DRAW.draw_landmarks(
                frame, hlm, _mp_hd.HAND_CONNECTIONS,
                MP_DRAW.DrawingSpec(color=C["phone"], thickness=2, circle_radius=2),
                MP_DRAW.DrawingSpec(color=(180,0,180), thickness=1),
            )
            # Simple phone heuristic: hand in upper frame near face
            wrist = hlm.landmark[0]
            if wrist.y < 0.6:
                hand_detected = True

    return face_bbox, face_lm, hand_detected


async def call_adk_agent(frame_b64: str) -> dict:
    """
    Call the ADK Coordinator Agent with a base64 frame.
    Returns parsed JSON dict with attention_score, alert_level, etc.
    """
    try:
        # Build runner for one-shot agent call
        runner = InMemoryRunner(
            agent=coordinator_agent,
            app_name="sleepguard"
        )
        session = await runner.session_service.create_session(
            app_name="sleepguard",
            user_id="user_1"
        )
        content = Content(
            role="user",
            parts=[Part(text=f"Analyze this frame: {frame_b64[:100]}...")]
        )
        # Stream events and get final response
        result_text = ""
        async for event in runner.run_async(
            user_id="user_1",
            session_id=session.id,
            new_message=content
        ):
            if event.is_final_response() and event.content:
                for part in event.content.parts:
                    if hasattr(part, 'text') and part.text:
                        result_text = part.text

        # Parse JSON from result
        result_text = re.sub(r"```json\s*|```\s*", "", result_text).strip()
        return json.loads(result_text)

    except Exception as e:
        # Fallback: call Gemini directly (simpler path)
        raw = analyze_frame_with_vision(frame_b64)
        vision = json.loads(raw)
        score_raw = compute_attention_score(
            vision.get("eyes_open", True),
            vision.get("yawning", False),
            vision.get("head_down", False),
            vision.get("head_sideways", False),
            vision.get("phone_visible", False),
        )
        result = json.loads(score_raw)
        result["observation"] = vision.get("observation", "")
        result.update(vision)
        return result


def apply_agent_result(s: SessionState, result: dict):
    """
    Merge ADK agent result into SessionState.
    Agent results override/boost MediaPipe-only scores.
    """
    s.agent_result  = result
    s.observation   = result.get("observation", s.observation)

    # Blend agent attention with mediapipe-derived score
    agent_att = result.get("attention_score", s.attention)
    s.attention = 0.6 * s.attention + 0.4 * agent_att   # weighted blend

    agent_slp = result.get("sleep_probability", s.sleep_prob)
    s.sleep_prob = 0.6 * s.sleep_prob + 0.4 * agent_slp

    s.alert_level = result.get("alert_level", s.alert_level)
    s.badges      = result.get("badges", s.badges)

    # Event tracking from agent
    is_sleeping = (s.eye_cf >= CFG["EAR_CONSEC"]) or (s.sleep_prob > 70)
    is_phone    = result.get("phone_visible", False)
    s.update_events(is_sleeping, is_phone)
    if is_phone: s.phone_eps = s.phone_eps  # already updated in update_events


def mediapipe_fallback_scores(s: SessionState):
    """
    Compute attention/sleep scores from MediaPipe only (no Gemini).
    Used on frames where agent is not called.
    """
    eye_s  = 0.0 if s.EAR >= CFG["EAR_THRESH"] else min(
        1.0, (CFG["EAR_THRESH"] - s.EAR) / 0.10)
    head_s = min(max(
        abs(s.pitch)/CFG["PITCH_T"],
        abs(s.yaw)/CFG["YAW_T"],
        abs(s.roll)/CFG["ROLL_T"]
    ), 1.0)
    yawn_s = min(s.yawn_cf / max(CFG["YAWN_CONSEC"],1), 1.0)

    d = CFG["W_EYE"]*eye_s + CFG["W_HEAD"]*head_s + CFG["W_YAWN"]*yawn_s
    s.attention  = round(100*(1-d), 1)
    s.sleep_prob = round(100*float(np.clip(0.55*eye_s+0.30*head_s+0.15*yawn_s,0,1)), 1)

    if s.eye_cf >= CFG["EAR_CONSEC"] or s.sleep_prob > 70:
        s.alert_level = "SLEEPING"
        if "SLEEPING" not in s.badges: s.badges = ["SLEEPING"]
    elif d > 0.55: s.alert_level = "DISTRACTED"
    elif d > 0.30: s.alert_level = "DROWSY"
    else:
        s.alert_level = "ATTENTIVE"
        s.badges = []

    s.update_events(s.alert_level=="SLEEPING", False)


def process_frame(b64_str: str) -> str:
    """
    Full pipeline for one frame:
    1. Decode base64 → OpenCV frame
    2. Run MediaPipe (every frame)
    3. Every CFG['AGENT_EVERY'] frames → call ADK Agent (Gemini)
    4. Render HUD
    5. Return base64 of processed frame
    """
    global SESSION
    s = SESSION

    # Decode
    frame = frame_from_b64(b64_str)
    if frame is None:
        return ""
    frame = cv2.flip(frame, 1)   # mirror (selfie mode)
    s.frame_num += 1
    s.tick_fps()

    # MediaPipe (fast, local, every frame)
    face_bbox, face_lm, hand_detected = run_mediapipe(frame, s)

    # ADK Agent every N frames (Gemini Vision)
    if s.frame_num % CFG["AGENT_EVERY"] == 0:
        try:
            # Encode frame for Gemini
            _, jpg = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
            frame_b64 = base64.b64encode(jpg).decode("utf-8")

            # Run agent (sync wrapper for async)
            loop = asyncio.get_event_loop()
            result = loop.run_until_complete(call_adk_agent(frame_b64))
            apply_agent_result(s, result)
        except Exception as e:
            # If agent fails, use MediaPipe scores only
            mediapipe_fallback_scores(s)
            s.observation = f"Agent err (using MP): {str(e)[:40]}"
    else:
        # Between agent calls: fast MediaPipe-only scoring
        mediapipe_fallback_scores(s)

    # Render HUD
    frame = render_hud(frame, s, face_bbox)

    return frame_to_b64(frame)


print("✅ Frame processor pipeline ready!")
print(f"   MediaPipe : runs EVERY frame (fast local detection)")
print(f"   ADK Agent : runs every {CFG['AGENT_EVERY']} frames (Gemini Vision)")
print(f"   Blend     : 60% MediaPipe + 40% Gemini for final scores")

---
## 🚀 CELL 9 — LAUNCH Real-Time Detection
> **RUN THIS CELL** to start SleepGuard AI.
>
> 1. Your browser will ask for **camera permission** — click Allow
> 2. The processed video with full HUD will appear below
> 3. **Gemini ADK Agent** analyzes every 10th frame
> 4. To stop: run the **STOP cell** (Cell 10) below

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 9 — 🚀 LAUNCH SleepGuard AI
#  Run this cell to start real-time detection
# ════════════════════════════════════════════════════════════
from IPython.display import display, HTML, Javascript, clear_output
import ipywidgets as widgets
import threading

# Reset session
SESSION = SessionState()

# Buffer for frame received from JS
_SG_FRAME_BUFFER = ""
_SG_RUNNING = True

# ── Display container ─────────────────────────────────────
out = widgets.Output()
display(out)

# ── Colab callback registration ───────────────────────────
try:
    from google.colab import output as colab_output
    def _colab_frame_cb(b64):
        global _SG_FRAME_BUFFER
        _SG_FRAME_BUFFER = b64
    colab_output.register_callback("notebook.process_frame", _colab_frame_cb)
    print("✅ Colab callback registered")
except Exception:
    pass  # Not on Colab; Kaggle uses kernel.execute instead

# ── Launch webcam JS ──────────────────────────────────────
display(Javascript(WEBCAM_JS))
print("📷 Webcam requested — check your browser for permission popup...")
print("⏳ Starting frame processing loop...")
time.sleep(2)  # give JS time to initialize camera

# ── Processing loop ───────────────────────────────────────
frame_count = 0
MAX = CFG["MAX_FRAMES"]

print("\n🟢 SleepGuard AI LIVE — processing frames...")
print("   Run Cell 10 to stop.\n")

while _SG_RUNNING:
    # ── Kaggle: poll the frame buffer ─────────────────────
    try:
        from IPython import get_ipython
        ip = get_ipython()
        # execute JS that writes base64 into Python variable
        buf_js = """
        (() => {
          if (window._sg_last_b64) {
            IPython.notebook.kernel.execute(
              `_SG_FRAME_BUFFER = '` + window._sg_last_b64 + `'`
            );
            window._sg_last_b64 = null;
          }
        })();
        """
    except Exception:
        pass

    # ── Wait for a frame to appear ────────────────────────
    if not _SG_FRAME_BUFFER:
        time.sleep(0.05)
        continue

    b64 = _SG_FRAME_BUFFER
    _SG_FRAME_BUFFER = ""

    # ── Process frame ─────────────────────────────────────
    try:
        out_b64 = process_frame(b64)
    except Exception as e:
        print(f"Frame error: {e}")
        continue

    if not out_b64:
        continue

    frame_count += 1

    # ── Display processed frame ───────────────────────────
    with out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style='background:#0a0f1e;padding:8px;border-radius:8px;
                    border:1px solid #1e3a5f;display:inline-block'>
          <img src='data:image/jpeg;base64,{out_b64}'
               style='border-radius:6px;display:block'/>
          <div style='color:#aaa;font-size:11px;font-family:monospace;
                      margin-top:4px;text-align:center'>
            🛡️ SleepGuard AI &nbsp;|&nbsp;
            Frame: {SESSION.frame_num} &nbsp;|&nbsp;
            Att: {SESSION.attention:.1f}% &nbsp;|&nbsp;
            Sleep: {SESSION.sleep_prob:.1f}% &nbsp;|&nbsp;
            {SESSION.alert_level}
          </div>
        </div>
        """))

    # Stop after MAX frames (0 = infinite)
    if MAX > 0 and frame_count >= MAX:
        print("\n✅ Reached MAX_FRAMES limit — stopping.")
        break

print("\n🔴 Detection stopped.")

---
## ⏹️ CELL 10 — Stop Detection

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 10 — Stop SleepGuard AI
# ════════════════════════════════════════════════════════════
_SG_RUNNING = False

# Stop webcam stream in browser
display(Javascript("""
  window._sg_active = false;
  if (window._sg_stream) {
    window._sg_stream.getTracks().forEach(t => t.stop());
    console.log('SleepGuard AI: webcam stopped');
  }
  if (window._sg_interval) clearInterval(window._sg_interval);
  const status = document.getElementById('sg_status');
  if (status) { status.textContent = '⏹️ Stopped'; status.style.color = '#ff4444'; }
"""))

print("\n⏹️ SleepGuard AI stopped.")
print("\n" + "═"*50)
print("📋 SESSION SUMMARY")
print("═"*50)
elapsed = int(time.time() - SESSION.t0)
summary = {
    "Total Frames Processed" : SESSION.frame_num,
    "Session Duration"       : f"{elapsed//60:02d}:{elapsed%60:02d}",
    "Avg Attention Score"    : f"{SESSION.attention:.1f}%",
    "Avg Sleep Probability"  : f"{SESSION.sleep_prob:.1f}%",
    "Total Blinks"           : SESSION.blinks,
    "Total Yawns"            : SESSION.yawns,
    "Sleep Episodes"         : SESSION.sleep_eps,
    "Phone Episodes"         : SESSION.phone_eps,
    "Total Sleep Time"       : f"{SESSION.sleep_secs:.1f}s",
    "Final Alert Level"      : SESSION.alert_level,
    "Last Gemini Observation": SESSION.observation,
}
for k, v in summary.items():
    print(f"  {k:<30}: {v}")

---
## 📊 CELL 11 — Session Analytics Dashboard

In [ ]:
# ════════════════════════════════════════════════════════════
#  CELL 11 — Post-Session Analytics
#  Run after stopping to see your session report
# ════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Sample gauges with final session data
att   = SESSION.attention
slp   = SESSION.sleep_prob
yawns = SESSION.yawns
blink = SESSION.blinks
slpep = SESSION.sleep_eps
phep  = SESSION.phone_eps
level = SESSION.alert_level

fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='#0a0f1e')
fig.suptitle('🛡️ SleepGuard AI — Session Report',
             color='white', fontsize=14, fontweight='bold')

# Color by alert level
level_colors = {
    "ATTENTIVE":  '#44cc44',
    "DROWSY":     '#ffcc00',
    "DISTRACTED": '#ff8800',
    "SLEEPING":   '#ff2244',
}
lc = level_colors.get(level, '#44cc44')

for ax in axes: ax.set_facecolor('#10182c')

# ── Gauge 1: Attention Score ──────────────────────────────
ax = axes[0]
ac = '#44cc44' if att>70 else '#ffcc00' if att>40 else '#ff2244'
wedges, _ = ax.pie(
    [att, 100-att], colors=[ac, '#1e2a40'],
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.35, edgecolor='#0a0f1e')
)
ax.text(0, 0, f"{att:.0f}%", ha='center', va='center',
        color='white', fontsize=18, fontweight='bold')
ax.set_title('Attention Score', color='white', fontsize=11)
ax.text(0, -1.5, 'Higher = better', ha='center', color='#aaa', fontsize=9)

# ── Gauge 2: Sleep Probability ────────────────────────────
ax = axes[1]
sc = '#44cc44' if slp<30 else '#ffcc00' if slp<60 else '#ff2244'
ax.pie(
    [slp, 100-slp], colors=[sc, '#1e2a40'],
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.35, edgecolor='#0a0f1e')
)
ax.text(0, 0, f"{slp:.0f}%", ha='center', va='center',
        color='white', fontsize=18, fontweight='bold')
ax.set_title('Sleep Probability', color='white', fontsize=11)
ax.text(0, -1.5, 'Lower = better', ha='center', color='#aaa', fontsize=9)

# ── Bar 3: Event Counts ───────────────────────────────────
ax = axes[2]
events   = ['Blinks', 'Yawns', 'Sleep\nEps', 'Phone\nEps']
vals     = [blink, yawns, slpep, phep]
ev_colors= ['#44aaff','#ffcc00','#ff2244','#cc00cc']
bars = ax.bar(events, vals, color=ev_colors, edgecolor='#0a0f1e', linewidth=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            str(v), ha='center', color='white', fontsize=10)
ax.set_title('Session Events', color='white', fontsize=11)
ax.tick_params(colors='white')
ax.spines[['top','right']].set_color('#333')
ax.spines[['left','bottom']].set_color('#333')
ax.set_facecolor('#10182c')

# ── Summary 4: Text Report ────────────────────────────────
ax = axes[3]
ax.axis('off')
elapsed = int(time.time() - SESSION.t0)
report = [
    ("Final Status",   level,        lc),
    ("Session Time",   f"{elapsed//60:02d}:{elapsed%60:02d}", 'white'),
    ("Frames",         str(SESSION.frame_num), 'white'),
    ("Attention",      f"{att:.1f}%",  ac),
    ("Sleep Prob",     f"{slp:.1f}%",  sc),
    ("Blinks",         str(blink),    '#44aaff'),
    ("Yawns",          str(yawns),    '#ffcc00'),
    ("Sleep Episodes", str(slpep),    '#ff2244'),
    ("Phone Episodes", str(phep),     '#cc00cc'),
]
ax.set_title('Summary', color='white', fontsize=11)
for i, (k, v, col) in enumerate(report):
    y = 0.92 - i * 0.10
    ax.text(0.05, y, k + ":", transform=ax.transAxes,
            color='#aaa', fontsize=9, va='top')
    ax.text(0.55, y, v, transform=ax.transAxes,
            color=col, fontsize=9, va='top', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/sleepguard_report.png', dpi=130,
            bbox_inches='tight', facecolor='#0a0f1e')
plt.show()
print("✅ Report saved: /tmp/sleepguard_report.png")

---

## 🗺️ Architecture Summary

| Component | Technology | Purpose |
|-----------|-----------|--------|
| **Webcam capture** | Browser JS + `getUserMedia()` | Real-time frame grab |
| **Frame transport** | Base64 + `kernel.execute()` | JS → Python bridge |
| **Face detection** | MediaPipe FaceMesh | EAR, MAR, head pose — every frame |
| **Hand detection** | MediaPipe Hands | Phone use detection |
| **AI Vision** | Gemini 2.0 Flash | Semantic analysis every 10 frames |
| **Agent orchestration** | Google ADK LlmAgent | Multi-agent pipeline |
| **Scoring** | Weighted formula | Attention 0-100%, Sleep 0-100% |
| **HUD overlay** | OpenCV drawing | Real-time display |
| **Analytics** | Matplotlib | Post-session report |

### 🧠 Why ADK Agents?

- **`VisionAnalyzerAgent`** — pure vision specialist, isolated and replaceable
- **`AlertAgent`** — pure scoring logic, can be swapped for ML model
- **`CoordinatorAgent`** — orchestrates both, maintains conversational context
- ADK handles retries, tool calling, session state, and streaming automatically
- You can add more agents (e.g. `NotificationAgent`, `ReportAgent`) with zero pipeline changes